# Coastal flood step 19: fixed 5,000 m mangrove attribution + distance-threshold diagnostics

This notebook does two things:

1. Computes nearest-mangrove distance for assets with positive avoided EAD and produces threshold tables.
2. Runs mangrove attribution using a fixed `5,000 m` buffer (equal split across nearby mangroves), so outlier distances do not set the buffer.

The fixed 5,000 m method is saved separately from step 07 outputs.


In [ ]:
from pathlib import Path

import numpy
import pandas
import geopandas
import matplotlib.pyplot as plt

pandas.set_option('display.max_columns', 220)
pandas.set_option('display.width', 240)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [ ]:
# User parameters
SCENARIO_FOR_ATTRIBUTION = 'minimum'  # 'minimum' or 'maximum'
SCENARIOS_FOR_THRESHOLDS = ['minimum', 'maximum']
FIXED_BUFFER_M = 5000.0

THRESHOLDS_M = [250, 500, 1000, 1500, 2000, 3000, 5000, 10000, 15000, 20000, 25000]

MAP_SCENARIO = SCENARIO_FOR_ATTRIBUTION
MAP_THRESHOLD_BINS_M = [250, 500, 1000, 1500, 2000, 3000, 5000, 10000, 15000, 20000, 25000]

if SCENARIO_FOR_ATTRIBUTION not in {'minimum', 'maximum'}:
    raise ValueError("SCENARIO_FOR_ATTRIBUTION must be 'minimum' or 'maximum'.")
for s in SCENARIOS_FOR_THRESHOLDS:
    if s not in {'minimum', 'maximum'}:
        raise ValueError(f'Invalid scenario in SCENARIOS_FOR_THRESHOLDS: {s}')
if MAP_SCENARIO not in SCENARIOS_FOR_THRESHOLDS:
    raise ValueError('MAP_SCENARIO must be included in SCENARIOS_FOR_THRESHOLDS.')

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
shared_intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
mangrove_path = base_path / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

for p in [network_csv, mangrove_path, jamaica_boundary_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print(f'Scenario for attribution: {SCENARIO_FOR_ATTRIBUTION}')
print(f'Fixed mangrove buffer (m): {FIXED_BUFFER_M:,.0f}')


In [ ]:
# Load shared inputs
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()

mangroves = geopandas.read_file(mangrove_path)
if mangroves.crs is None:
    raise ValueError('Mangrove CRS is missing.')
if str(mangroves.crs).upper() != 'EPSG:3448':
    mangroves = mangroves.to_crs('EPSG:3448')

if 'ID' in mangroves.columns:
    mangroves['Mangrove_ID'] = mangroves['ID'].astype(int)
else:
    mangroves['Mangrove_ID'] = numpy.arange(1, len(mangroves) + 1)

mangrove_base_cols = ['Mangrove_ID']
for c in ['Parish', 'HECTARES', 'TYPE']:
    if c in mangroves.columns:
        mangrove_base_cols.append(c)

jamaica_boundary = geopandas.read_file(jamaica_boundary_path)
if jamaica_boundary.crs is None:
    raise ValueError('Jamaica boundary CRS is missing.')
if str(jamaica_boundary.crs).upper() != 'EPSG:3448':
    jamaica_boundary = jamaica_boundary.to_crs('EPSG:3448')

print(f'Mangrove patches: {len(mangroves):,}')
print('Network layers in metadata:', len(network_details))


In [ ]:
# Helper: build one geometry per asset with avoided EAD
asset_key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']

def build_asset_gdf_for_scenario(scenario):
    results_path = base_path / f'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_{scenario}_scenario'
    asset_out = results_path / 'damage_estimates' / 'coastal_ead_asset_level_usd.csv'
    if not asset_out.exists():
        raise FileNotFoundError(f'Missing asset-level EAD file: {asset_out}')

    asset_ead = pandas.read_csv(asset_out)

    map_layers = []
    missing_split_files = []

    for row in network_details.itertuples(index=False):
        split_file = shared_intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
        if not split_file.exists():
            missing_split_files.append(str(split_file))
            continue

        split_geom = geopandas.read_parquet(split_file)
        if split_geom.crs is not None:
            split_geom = split_geom.to_crs('EPSG:3448')
        if row.asset_id_column not in split_geom.columns:
            continue

        split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
        split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs='EPSG:3448')

        ead_subset = asset_ead.loc[
            (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
            ['Asset_ID', 'Avoided_EAD_USD']
        ].copy()

        if ead_subset.empty:
            continue

        split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
        ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

        merged = split_geom.merge(
            ead_subset[['_join_id', 'Avoided_EAD_USD']],
            on='_join_id',
            how='left'
        )

        merged['Sector'] = row.sector
        merged['Subsector'] = row.asset_description
        merged['Asset'] = row.asset_gpkg
        merged['Layer'] = row.asset_layer
        merged['Asset_ID'] = merged[row.asset_id_column].astype(str)
        merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

        map_layers.append(merged[asset_key_cols + ['Avoided_EAD_USD', 'geometry']])

    if not map_layers:
        raise ValueError(f'No map layers could be built for scenario: {scenario}')

    asset_gdf = geopandas.GeoDataFrame(pandas.concat(map_layers, ignore_index=True), geometry='geometry', crs='EPSG:3448')
    asset_gdf = asset_gdf.dissolve(
        by=asset_key_cols,
        as_index=False,
        aggfunc={'Avoided_EAD_USD': 'first'}
    )
    asset_gdf = geopandas.GeoDataFrame(asset_gdf, geometry='geometry', crs='EPSG:3448')

    out = {
        'results_path': results_path,
        'asset_gdf': asset_gdf,
        'missing_split_files': sorted(set(missing_split_files)),
        'asset_ead_rows': len(asset_ead),
    }
    return out


In [ ]:
# Build scenario asset geodataframes once
scenario_data = {}
for scenario in SCENARIOS_FOR_THRESHOLDS:
    d = build_asset_gdf_for_scenario(scenario)
    scenario_data[scenario] = d
    print(f"{scenario}: assets={len(d['asset_gdf']):,}, EAD rows={d['asset_ead_rows']:,}, missing split files={len(d['missing_split_files'])}")


In [ ]:
# Distance-threshold calculations (positive avoided EAD assets only)
threshold_tables = {}
bin_tables = {}
nearest_assets_by_scenario = {}

for scenario in SCENARIOS_FOR_THRESHOLDS:
    asset_gdf = scenario_data[scenario]['asset_gdf']
    results_path = scenario_data[scenario]['results_path']

    positive_assets = asset_gdf.loc[asset_gdf['Avoided_EAD_USD'] > 0, asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()
    if positive_assets.empty:
        print(f'{scenario}: no assets with positive avoided EAD')
        continue

    nearest = geopandas.sjoin_nearest(
        positive_assets,
        mangroves[['Mangrove_ID', 'geometry']],
        how='left',
        distance_col='nearest_mangrove_distance_m'
    )
    nearest = nearest.dropna(subset=['nearest_mangrove_distance_m']).copy()

    nearest_assets_by_scenario[scenario] = nearest

    total_assets = len(nearest)
    total_avoided = float(nearest['Avoided_EAD_USD'].sum())

    # Cumulative threshold table
    threshold_rows = []
    for t in THRESHOLDS_M:
        within = nearest.loc[nearest['nearest_mangrove_distance_m'] <= float(t)]
        n = int(len(within))
        a = float(within['Avoided_EAD_USD'].sum())
        threshold_rows.append({
            'Scenario': scenario,
            'Threshold_m': float(t),
            'Assets_within_n': n,
            'Assets_within_pct': 100.0 * n / total_assets if total_assets > 0 else numpy.nan,
            'Avoided_EAD_within_USD': a,
            'Avoided_EAD_within_pct': 100.0 * a / total_avoided if total_avoided > 0 else numpy.nan,
        })

    threshold_df = pandas.DataFrame(threshold_rows)
    threshold_tables[scenario] = threshold_df

    # Bin table across the same threshold edges
    max_dist = float(nearest['nearest_mangrove_distance_m'].max())
    bin_edges = [0.0] + [float(t) for t in THRESHOLDS_M] + [float(numpy.ceil(max_dist))]
    clean_edges = [bin_edges[0]]
    for e in bin_edges[1:]:
        if e > clean_edges[-1]:
            clean_edges.append(e)

    bins = pandas.IntervalIndex.from_breaks(clean_edges, closed='right')
    nearest_for_bins = nearest.copy()
    nearest_for_bins['distance_bin'] = pandas.cut(nearest_for_bins['nearest_mangrove_distance_m'], bins=bins)

    bin_df = nearest_for_bins.groupby('distance_bin', observed=False).agg(
        Asset_count=('Asset_ID', 'size'),
        Avoided_EAD_USD=('Avoided_EAD_USD', 'sum')
    ).reset_index()
    bin_df = bin_df.loc[bin_df['Asset_count'] > 0].copy()
    bin_df['Scenario'] = scenario
    bin_df['Asset_pct'] = 100.0 * bin_df['Asset_count'] / total_assets
    bin_df['Avoided_EAD_pct'] = 100.0 * bin_df['Avoided_EAD_USD'] / total_avoided if total_avoided > 0 else numpy.nan
    bin_df['distance_bin'] = bin_df['distance_bin'].astype(str)
    bin_tables[scenario] = bin_df

    # Save threshold outputs in scenario-specific folder
    out_dir = results_path / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
    out_dir.mkdir(parents=True, exist_ok=True)
    threshold_df.to_csv(out_dir / 'distance_threshold_distribution_positive_avoided_assets.csv', index=False)
    bin_df.to_csv(out_dir / 'distance_bin_distribution_positive_avoided_assets.csv', index=False)

    print(f"{scenario}: positive assets={total_assets:,}, average distance={nearest['nearest_mangrove_distance_m'].mean():,.2f} m, max distance={max_dist:,.2f} m")


In [ ]:
# Display threshold tables
for scenario in SCENARIOS_FOR_THRESHOLDS:
    if scenario not in threshold_tables:
        continue
    print('\n' + '=' * 95)
    print(f'SCENARIO: {scenario.upper()} | Cumulative distance thresholds (positive avoided EAD assets)')
    print('=' * 95)
    display(
        threshold_tables[scenario][[
            'Threshold_m', 'Assets_within_n', 'Assets_within_pct',
            'Avoided_EAD_within_USD', 'Avoided_EAD_within_pct'
        ]]
    )


In [ ]:
# Geospatial view of distance-threshold classes for MAP_SCENARIO
if MAP_SCENARIO not in nearest_assets_by_scenario:
    raise ValueError(f'No nearest-distance data for MAP_SCENARIO={MAP_SCENARIO}')

plot_assets = nearest_assets_by_scenario[MAP_SCENARIO].copy()
plot_assets = geopandas.GeoDataFrame(plot_assets, geometry='geometry', crs='EPSG:3448')
plot_assets['geometry'] = plot_assets.geometry.representative_point()

map_edges = [0.0] + [float(x) for x in MAP_THRESHOLD_BINS_M] + [numpy.inf]
map_labels = []
for i in range(1, len(map_edges) - 1):
    left = int(map_edges[i - 1])
    right = int(map_edges[i])
    map_labels.append(f'{left}-{right} m')
map_labels.append(f'>{int(MAP_THRESHOLD_BINS_M[-1])} m')

plot_assets['distance_band'] = pandas.cut(
    plot_assets['nearest_mangrove_distance_m'],
    bins=map_edges,
    labels=map_labels,
    include_lowest=True,
    right=True,
)

# Ensure legend order follows thresholds
plot_assets['distance_band'] = pandas.Categorical(plot_assets['distance_band'], categories=map_labels, ordered=True)

fig, ax = plt.subplots(figsize=(11, 9))
ax.set_facecolor('white')
jamaica_boundary.boundary.plot(ax=ax, color='#8e8e8e', linewidth=0.35, zorder=1)
mangroves.boundary.plot(ax=ax, color='#444444', linewidth=0.15, alpha=0.35, zorder=2)

plot_assets.plot(
    ax=ax,
    column='distance_band',
    categorical=True,
    markersize=5,
    alpha=0.85,
    legend=True,
    legend_kwds={'title': 'Nearest mangrove distance'},
    zorder=3,
)

ax.set_title(
    f'Assets with positive avoided EAD by nearest-mangrove distance band ({MAP_SCENARIO} scenario)',
    fontsize=12,
)
ax.set_axis_off()
plt.tight_layout()

map_out_dir = scenario_data[MAP_SCENARIO]['results_path'] / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
map_out_dir.mkdir(parents=True, exist_ok=True)
out_png = map_out_dir / f'assets_distance_threshold_bands_{MAP_SCENARIO}.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()


In [ ]:
# Fixed 5,000 m mangrove attribution for SCENARIO_FOR_ATTRIBUTION
if SCENARIO_FOR_ATTRIBUTION not in scenario_data:
    scenario_data[SCENARIO_FOR_ATTRIBUTION] = build_asset_gdf_for_scenario(SCENARIO_FOR_ATTRIBUTION)

asset_gdf_attr = scenario_data[SCENARIO_FOR_ATTRIBUTION]['asset_gdf'].copy()
results_path_attr = scenario_data[SCENARIO_FOR_ATTRIBUTION]['results_path']

mangrove_buffers = mangroves[mangrove_base_cols + ['geometry']].copy()
mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(FIXED_BUFFER_M)

joined = geopandas.sjoin(
    asset_gdf_attr,
    mangrove_buffers[['Mangrove_ID', 'geometry']],
    how='left',
    predicate='intersects',
)

joined['nearby_mangrove_count'] = joined.groupby(asset_key_cols)['Mangrove_ID'].transform(lambda s: s.notna().sum())
joined['nearby_mangrove_count'] = joined['nearby_mangrove_count'].fillna(0).astype(int)

joined['Avoided_EAD_USD_attributed'] = numpy.where(
    (joined['Mangrove_ID'].notna()) & (joined['nearby_mangrove_count'] > 0),
    joined['Avoided_EAD_USD'] / joined['nearby_mangrove_count'],
    0.0,
)

asset_unique = asset_gdf_attr[asset_key_cols + ['Avoided_EAD_USD']].copy()
total_avoided_usd = float(asset_unique['Avoided_EAD_USD'].sum())

matched_assets = joined.loc[joined['nearby_mangrove_count'] > 0, asset_key_cols].drop_duplicates()
matched_assets['matched'] = 1
asset_match_status = asset_unique.merge(matched_assets, on=asset_key_cols, how='left')
asset_match_status['matched'] = asset_match_status['matched'].fillna(0).astype(int)

attributed_total_usd = float(joined['Avoided_EAD_USD_attributed'].sum())
unattributed_total_usd = float(asset_match_status.loc[asset_match_status['matched'] == 0, 'Avoided_EAD_USD'].sum())

joined_m = joined.dropna(subset=['Mangrove_ID']).copy()
joined_m['Mangrove_ID'] = joined_m['Mangrove_ID'].astype(int)

mangrove_sector_summary = (
    joined_m.groupby(['Mangrove_ID', 'Sector'], as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .sort_values(['Mangrove_ID', 'Sector'])
)

mangrove_total_summary = (
    joined_m.groupby('Mangrove_ID', as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .rename(columns={'Avoided_EAD_USD_attributed': 'Total_Avoided_EAD_USD_attributed'})
    .sort_values('Total_Avoided_EAD_USD_attributed', ascending=False)
)

mangrove_attribution_map = mangroves[mangrove_base_cols + ['geometry']].merge(
    mangrove_total_summary,
    on='Mangrove_ID',
    how='left'
)
mangrove_attribution_map['Total_Avoided_EAD_USD_attributed'] = mangrove_attribution_map['Total_Avoided_EAD_USD_attributed'].fillna(0.0)

print(f'Scenario: {SCENARIO_FOR_ATTRIBUTION}')
print(f'Fixed attribution buffer (m): {FIXED_BUFFER_M:,.0f}')
print(f'Total avoided EAD across assets (USD): {total_avoided_usd:,.2f}')
print(f'Total attributed to mangroves (USD): {attributed_total_usd:,.2f}')
print(f'Total not attributed (outside buffer) (USD): {unattributed_total_usd:,.2f}')
if abs(total_avoided_usd) > 0:
    print(f'Fraction attributed: {100 * attributed_total_usd / total_avoided_usd:,.2f}%')


In [ ]:
# Save fixed 5,000 m attribution outputs
out_dir_attr = results_path_attr / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
out_dir_attr.mkdir(parents=True, exist_ok=True)

joined_out = joined[[
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD',
    'Mangrove_ID', 'nearby_mangrove_count', 'Avoided_EAD_USD_attributed'
]].copy()

summary_out = pandas.DataFrame([
    {
        'Scenario': SCENARIO_FOR_ATTRIBUTION,
        'Fixed_Buffer_m': FIXED_BUFFER_M,
        'Total_Avoided_EAD_USD': total_avoided_usd,
        'Attributed_Total_USD': attributed_total_usd,
        'Unattributed_Total_USD': unattributed_total_usd,
        'Fraction_Attributed_pct': (100.0 * attributed_total_usd / total_avoided_usd) if abs(total_avoided_usd) > 0 else numpy.nan,
    }
])

joined_out.to_csv(out_dir_attr / 'asset_to_nearby_mangrove_attribution_fixed_5000m.csv', index=False)
mangrove_sector_summary.to_csv(out_dir_attr / 'mangrove_attribution_by_sector_fixed_5000m.csv', index=False)
mangrove_total_summary.to_csv(out_dir_attr / 'mangrove_attribution_total_fixed_5000m.csv', index=False)
summary_out.to_csv(out_dir_attr / 'run_summary_fixed_5000m.csv', index=False)
mangrove_attribution_map.to_file(out_dir_attr / 'mangrove_attribution_total_fixed_5000m.gpkg', driver='GPKG')

# Optional comparison with the fixed 1000 m output from step 07
fixed_1000_csv = results_path_attr / 'damage_estimates' / 'mangrove_attribution' / 'mangrove_attribution_total_1000m.csv'
if fixed_1000_csv.exists():
    fixed_1000 = pandas.read_csv(fixed_1000_csv)
    fixed_1000_attributed = float(fixed_1000['Total_Avoided_EAD_USD_attributed'].sum())

    compare = pandas.DataFrame([
        {
            'Method': 'Fixed 1000m (step 07)',
            'Buffer_m': 1000.0,
            'Attributed_Total_USD': fixed_1000_attributed,
            'Unattributed_Total_USD': total_avoided_usd - fixed_1000_attributed,
            'Fraction_Attributed_pct': 100.0 * fixed_1000_attributed / total_avoided_usd if abs(total_avoided_usd) > 0 else numpy.nan,
        },
        {
            'Method': 'Fixed 5000m (this notebook)',
            'Buffer_m': FIXED_BUFFER_M,
            'Attributed_Total_USD': attributed_total_usd,
            'Unattributed_Total_USD': unattributed_total_usd,
            'Fraction_Attributed_pct': 100.0 * attributed_total_usd / total_avoided_usd if abs(total_avoided_usd) > 0 else numpy.nan,
        },
    ])

    compare.to_csv(out_dir_attr / 'comparison_fixed_1000m_vs_5000m.csv', index=False)
    print('Saved comparison to:')
    print(out_dir_attr / 'comparison_fixed_1000m_vs_5000m.csv')
    display(compare)
else:
    print(f'No fixed 1000m baseline found at: {fixed_1000_csv}')

print(f'Saved fixed 5000m outputs to: {out_dir_attr}')
print('Top 10 mangroves by attributed avoided EAD (USD):')
display(mangrove_total_summary.head(10))


In [ ]:
# Map: mangrove-attributed avoided EAD for fixed 5,000 m buffer
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

if 'mangrove_attribution_map' not in globals():
    raise ValueError('Run the fixed 5,000 m attribution cell first.')
if 'out_dir_attr' not in globals():
    out_dir_attr = results_path_attr / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
    out_dir_attr.mkdir(parents=True, exist_ok=True)

map_gdf = mangrove_attribution_map.copy()
if map_gdf.crs is None or str(map_gdf.crs).upper() != 'EPSG:3448':
    map_gdf = map_gdf.to_crs('EPSG:3448')

value_col = 'Total_Avoided_EAD_USD_attributed'
vals = map_gdf[value_col].fillna(0.0)
abs_vals = vals.abs()
true_max_abs = float(abs_vals.max())

# Larger canvas + stronger contrast so patch colours are easier to distinguish
map_figsize = (16, 13)
display_quantile = 0.95

display_cap = float(abs_vals.quantile(display_quantile))
if display_cap <= 0:
    display_cap = true_max_abs if true_max_abs > 0 else 1.0

map_gdf['_plot_val'] = vals.clip(-display_cap, display_cap)

cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)
norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

fig, ax = plt.subplots(figsize=map_figsize)
ax.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

map_gdf.plot(
    ax=ax,
    column='_plot_val',
    cmap=cmap,
    norm=norm,
    linewidth=0.35,
    edgecolor='#4f4f4f',
    alpha=0.98,
    zorder=2,
)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label(
    f'Attributed avoided EAD (USD), clipped at q={display_quantile:.2f} | Red=increase, White=no change, Green=avoided',
    rotation=90,
)

ax.set_title(
    f'Fixed {FIXED_BUFFER_M:,.0f} m buffer: mangrove-attributed avoided EAD ({SCENARIO_FOR_ATTRIBUTION} scenario) | true max abs={true_max_abs:,.2f}',
    fontsize=14,
)
ax.set_axis_off()
plt.tight_layout()

out_png = out_dir_attr / f'mangrove_attribution_map_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()


In [ ]:
# Overlay map: unattributed avoided-EAD assets and zero-attribution mangroves
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

if 'asset_gdf_attr' not in globals() or 'asset_match_status' not in globals():
    raise ValueError('Run the fixed 5,000 m attribution cell first so asset match status exists.')
if 'mangrove_attribution_map' not in globals():
    raise ValueError('Run the fixed 5,000 m attribution cell first so mangrove_attribution_map exists.')
if 'out_dir_attr' not in globals():
    out_dir_attr = results_path_attr / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
    out_dir_attr.mkdir(parents=True, exist_ok=True)

value_col = 'Total_Avoided_EAD_USD_attributed'

asset_match_cols = asset_key_cols + ['matched']
asset_plot = asset_gdf_attr[asset_key_cols + ['Avoided_EAD_USD', 'geometry']].merge(
    asset_match_status[asset_match_cols],
    on=asset_key_cols,
    how='left'
)
asset_plot['matched'] = asset_plot['matched'].fillna(0).astype(int)

# Assets with positive avoided EAD that were not attributed to any mangrove under the 5,000 m rule
unattributed_positive_assets = asset_plot.loc[
    (asset_plot['Avoided_EAD_USD'] > 0) & (asset_plot['matched'] == 0)
].copy()
unattributed_positive_assets['geometry'] = unattributed_positive_assets.geometry.representative_point()

mangrove_plot = mangrove_attribution_map.copy()
if mangrove_plot.crs is None or str(mangrove_plot.crs).upper() != 'EPSG:3448':
    mangrove_plot = mangrove_plot.to_crs('EPSG:3448')

zero_mask = mangrove_plot[value_col].fillna(0.0).abs() <= 1e-9
zero_attribution_mangroves = mangrove_plot.loc[zero_mask].copy()
nonzero_attribution_mangroves = mangrove_plot.loc[~zero_mask].copy()

print(f'Unattributed assets with positive avoided EAD: {len(unattributed_positive_assets):,}')
print(f'Unattributed positive avoided EAD total (USD): {float(unattributed_positive_assets["Avoided_EAD_USD"].sum()):,.2f}')
print(f'Mangrove patches with zero attributed EAD: {len(zero_attribution_mangroves):,} / {len(mangrove_plot):,}')

fig, ax = plt.subplots(figsize=(16, 13))
ax.set_facecolor('white')
jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

# Context: mangroves receiving some attributed value
nonzero_attribution_mangroves.plot(
    ax=ax,
    facecolor='#4daf4a',
    edgecolor='#3f3f3f',
    linewidth=0.25,
    alpha=0.35,
    zorder=2,
)

# Highlight: mangroves with no attributed damages
zero_attribution_mangroves.plot(
    ax=ax,
    facecolor='#f39c12',
    edgecolor='#2f2f2f',
    linewidth=0.35,
    alpha=0.90,
    zorder=3,
)

# Highlight: positive avoided-EAD assets not attributed
if len(unattributed_positive_assets) > 0:
    unattributed_positive_assets.plot(
        ax=ax,
        color='#ff00aa',
        markersize=18,
        alpha=0.95,
        zorder=4,
    )

legend_handles = [
    Patch(facecolor='#4daf4a', edgecolor='#3f3f3f', alpha=0.35, label='Mangroves with attributed EAD'),
    Patch(facecolor='#f39c12', edgecolor='#2f2f2f', alpha=0.90, label='Mangroves with zero attributed EAD'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff00aa', markeredgecolor='#ff00aa', markersize=8, label='Unattributed assets (Avoided_EAD_USD > 0)')
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, facecolor='white', framealpha=0.95)

ax.set_title(
    f'Fixed {FIXED_BUFFER_M:,.0f} m buffer: unattributed avoided-EAD assets and zero-attribution mangroves ({SCENARIO_FOR_ATTRIBUTION} scenario)',
    fontsize=14,
)
ax.set_axis_off()
plt.tight_layout()

out_png = out_dir_attr / f'mangrove_zero_attribution_and_unattributed_assets_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()


In [ ]:
# Maps: closest mangroves to unattributed avoided-EAD assets (all + north coast)
from shapely.geometry import LineString
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

if 'asset_gdf_attr' not in globals() or 'asset_match_status' not in globals():
    raise ValueError('Run the fixed 5,000 m attribution cell first.')
if 'mangroves' not in globals() or 'jamaica_boundary' not in globals():
    raise ValueError('Run earlier setup cells so mangroves and boundary are loaded.')
if 'out_dir_attr' not in globals():
    out_dir_attr = results_path_attr / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
    out_dir_attr.mkdir(parents=True, exist_ok=True)

asset_match_cols = asset_key_cols + ['matched']
asset_plot = asset_gdf_attr[asset_key_cols + ['Avoided_EAD_USD', 'geometry']].merge(
    asset_match_status[asset_match_cols],
    on=asset_key_cols,
    how='left'
)
asset_plot['matched'] = asset_plot['matched'].fillna(0).astype(int)

# Unattributed assets with positive avoided EAD
unattributed_all = asset_plot.loc[
    (asset_plot['Avoided_EAD_USD'] > 0) & (asset_plot['matched'] == 0)
].copy()
if unattributed_all.empty:
    raise ValueError('No unattributed assets with positive avoided EAD found.')

unattributed_all['geometry'] = unattributed_all.geometry.representative_point()

# Nearest FoN mangrove patch for each unattributed asset
nearest_all = geopandas.sjoin_nearest(
    unattributed_all,
    mangroves[['Mangrove_ID', 'geometry']],
    how='left',
    distance_col='nearest_mangrove_distance_m'
)
nearest_all = nearest_all.drop(columns=['index_right'])

closest_ids_all = sorted(set(nearest_all['Mangrove_ID'].dropna().astype(int).tolist()))
closest_mang_all = mangroves[mangroves['Mangrove_ID'].isin(closest_ids_all)].copy()

# Build connection lines (asset -> nearest mangrove centroid)
mang_cent_all = closest_mang_all[['Mangrove_ID', 'geometry']].copy()
mang_cent_all['geometry'] = mang_cent_all.geometry.centroid
conn_all = nearest_all.merge(
    mang_cent_all.rename(columns={'geometry': 'mang_centroid'}),
    on='Mangrove_ID',
    how='left'
)
line_geoms_all = []
for row in conn_all.itertuples(index=False):
    if row.geometry is None or row.mang_centroid is None:
        line_geoms_all.append(None)
    else:
        line_geoms_all.append(LineString([row.geometry, row.mang_centroid]))

lines_all = geopandas.GeoDataFrame(
    conn_all[asset_key_cols + ['Avoided_EAD_USD', 'Mangrove_ID', 'nearest_mangrove_distance_m']].copy(),
    geometry=line_geoms_all,
    crs='EPSG:3448'
).dropna(subset=['geometry'])

# ---------- Map A: all unattributed assets + closest mangroves ----------
fig, ax = plt.subplots(figsize=(16, 13))
ax.set_facecolor('white')
jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

# Context mangroves (grey)
mangroves.plot(ax=ax, facecolor='#d9d9d9', edgecolor='#8c8c8c', linewidth=0.20, alpha=0.55, zorder=2)

# Closest mangroves (blue)
closest_mang_all.plot(ax=ax, facecolor='#007bff', edgecolor='#0b1f3a', linewidth=0.45, alpha=0.95, zorder=3)

# Connection lines
if len(lines_all) > 0:
    lines_all.plot(ax=ax, color='#ff7f0e', linewidth=0.35, alpha=0.25, zorder=4)

# Assets
asset_sizes = 16 + 10 * numpy.log10(numpy.clip(nearest_all['Avoided_EAD_USD'].values, 1e-6, None) + 1)
nearest_all.plot(ax=ax, color='#ff00aa', markersize=asset_sizes, alpha=0.90, zorder=5)

legend_handles_all = [
    Patch(facecolor='#d9d9d9', edgecolor='#8c8c8c', label='All FoN mangroves (context)'),
    Patch(facecolor='#007bff', edgecolor='#0b1f3a', label='Closest mangroves to unattributed assets'),
    Line2D([0], [0], color='#ff7f0e', lw=1.2, label='Asset -> closest mangrove link'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff00aa', markeredgecolor='#ff00aa', markersize=8, label='Unattributed assets (Avoided_EAD_USD > 0)')
]
ax.legend(handles=legend_handles_all, loc='lower left', frameon=True, facecolor='white', framealpha=0.95)
ax.set_title(
    f'All unattributed positive avoided-EAD assets and closest FoN mangroves ({SCENARIO_FOR_ATTRIBUTION} scenario) | fixed {FIXED_BUFFER_M:,.0f} m buffer',
    fontsize=14,
)
ax.set_axis_off()
plt.tight_layout()

all_map_png = out_dir_attr / f'all_unattributed_assets_and_closest_mangroves_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png'
fig.savefig(all_map_png, dpi=300, bbox_inches='tight')
print(f'Saved: {all_map_png}')
plt.show()

# ---------- North-coast subset ----------
coastline = jamaica_boundary.geometry.iloc[0].boundary
nearest_all['dist_to_coast_m'] = nearest_all.geometry.distance(coastline)

nearest_all_ll = nearest_all.to_crs('EPSG:4326')
nearest_all['lat'] = nearest_all_ll.geometry.y
north_lat_threshold = float(jamaica_boundary.to_crs('EPSG:4326').geometry.iloc[0].centroid.y)

north_coast = nearest_all.loc[
    (nearest_all['dist_to_coast_m'] <= 500.0) & (nearest_all['lat'] >= north_lat_threshold)
].copy()

print(f'North-coast subset count: {len(north_coast):,}')
if len(north_coast) > 0:
    print(f'North-coast avoided EAD sum (USD): {float(north_coast["Avoided_EAD_USD"].sum()):,.2f}')
    print('North-coast avoided EAD distribution (USD):')
    display(north_coast['Avoided_EAD_USD'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

    nearest_ids_north = sorted(set(north_coast['Mangrove_ID'].dropna().astype(int).tolist()))
    closest_mang_north = mangroves[mangroves['Mangrove_ID'].isin(nearest_ids_north)].copy()

    mang_cent_north = closest_mang_north[['Mangrove_ID', 'geometry']].copy()
    mang_cent_north['geometry'] = mang_cent_north.geometry.centroid
    conn_north = north_coast.merge(
        mang_cent_north.rename(columns={'geometry': 'mang_centroid'}),
        on='Mangrove_ID',
        how='left'
    )

    line_geoms_north = []
    for row in conn_north.itertuples(index=False):
        if row.geometry is None or row.mang_centroid is None:
            line_geoms_north.append(None)
        else:
            line_geoms_north.append(LineString([row.geometry, row.mang_centroid]))

    lines_north = geopandas.GeoDataFrame(
        conn_north[asset_key_cols + ['Avoided_EAD_USD', 'Mangrove_ID', 'nearest_mangrove_distance_m']].copy(),
        geometry=line_geoms_north,
        crs='EPSG:3448'
    ).dropna(subset=['geometry'])

    fig, ax = plt.subplots(figsize=(16, 13))
    ax.set_facecolor('white')
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)
    mangroves.plot(ax=ax, facecolor='#d9d9d9', edgecolor='#8c8c8c', linewidth=0.20, alpha=0.55, zorder=2)
    closest_mang_north.plot(ax=ax, facecolor='#00c2ff', edgecolor='#0d2a33', linewidth=0.45, alpha=0.95, zorder=3)

    if len(lines_north) > 0:
        lines_north.plot(ax=ax, color='#ff7f0e', linewidth=0.45, alpha=0.35, zorder=4)

    sizes_north = 18 + 10 * numpy.log10(numpy.clip(north_coast['Avoided_EAD_USD'].values, 1e-6, None) + 1)
    north_coast.plot(ax=ax, color='#ff00aa', markersize=sizes_north, alpha=0.95, zorder=5)

    legend_handles_north = [
        Patch(facecolor='#d9d9d9', edgecolor='#8c8c8c', label='All FoN mangroves (context)'),
        Patch(facecolor='#00c2ff', edgecolor='#0d2a33', label='Closest mangroves to north-coast subset'),
        Line2D([0], [0], color='#ff7f0e', lw=1.2, label='Asset -> closest mangrove link'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff00aa', markeredgecolor='#ff00aa', markersize=8, label='North-coast unattributed assets')
    ]
    ax.legend(handles=legend_handles_north, loc='lower left', frameon=True, facecolor='white', framealpha=0.95)

    ax.set_title(
        f'North-coast unattributed assets and closest FoN mangroves ({SCENARIO_FOR_ATTRIBUTION} scenario) | fixed {FIXED_BUFFER_M:,.0f} m buffer',
        fontsize=14,
    )
    ax.set_axis_off()
    plt.tight_layout()

    north_map_png = out_dir_attr / f'north_coast_unattributed_assets_and_closest_mangroves_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png'
    fig.savefig(north_map_png, dpi=300, bbox_inches='tight')
    print(f'Saved: {north_map_png}')
    plt.show()
else:
    print('North-coast subset is empty under current definition.')


In [ ]:
# Numbered map + table: closest mangroves ranked by unattributed avoided EAD
from shapely.geometry import LineString
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

if 'nearest_all' not in globals() or 'closest_mang_all' not in globals():
    raise ValueError('Run the closest-mangroves map cell first so nearest_all/closest_mang_all exist.')
if 'out_dir_attr' not in globals():
    out_dir_attr = results_path_attr / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
    out_dir_attr.mkdir(parents=True, exist_ok=True)

# Aggregate unattributed avoided EAD by nearest mangrove patch
nearest_summary = (
    nearest_all.dropna(subset=['Mangrove_ID'])
    .assign(Mangrove_ID=lambda d: d['Mangrove_ID'].astype(int))
    .groupby('Mangrove_ID', as_index=False)
    .agg(
        Unattributed_Avoided_EAD_USD=('Avoided_EAD_USD', 'sum'),
        Unattributed_Asset_Count=('Asset_ID', 'size'),
        Mean_Asset_Distance_to_Mangrove_m=('nearest_mangrove_distance_m', 'mean')
    )
    .sort_values('Unattributed_Avoided_EAD_USD', ascending=False)
    .reset_index(drop=True)
)
nearest_summary['Label_No'] = nearest_summary.index + 1

mang_label = closest_mang_all.merge(nearest_summary, on='Mangrove_ID', how='left')

# Coast-side classification by mangrove centroid latitude relative to Jamaica centroid latitude
jamaica_centroid_lat = float(jamaica_boundary.to_crs('EPSG:4326').geometry.iloc[0].centroid.y)
mang_label_ll = mang_label.to_crs('EPSG:4326').copy()
mang_label_ll['centroid_lat'] = mang_label_ll.geometry.centroid.y
mang_label_ll['Coast_Side'] = numpy.where(mang_label_ll['centroid_lat'] >= jamaica_centroid_lat, 'North', 'South')

# Bring coast-side flag back to projected frame
mang_label = mang_label.merge(
    mang_label_ll[['Mangrove_ID', 'Coast_Side', 'centroid_lat']],
    on='Mangrove_ID',
    how='left'
)

# Reference table for map crosswalk
cols_order = [
    'Label_No', 'Mangrove_ID', 'Coast_Side', 'Parish',
    'Unattributed_Asset_Count', 'Unattributed_Avoided_EAD_USD',
    'Mean_Asset_Distance_to_Mangrove_m', 'centroid_lat'
]
ref_table = mang_label[cols_order].sort_values('Label_No').copy()

print('Closest mangroves used by unattributed assets:')
display(ref_table)

ref_csv = out_dir_attr / f'closest_mangroves_unattributed_reference_table_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.csv'
ref_table.to_csv(ref_csv, index=False)
print(f'Saved reference table: {ref_csv}')

# Build asset -> mangrove centroid lines so mapping is explicit
mang_cent = mang_label[['Mangrove_ID', 'geometry']].copy()
mang_cent['geometry'] = mang_cent.geometry.centroid
conn = nearest_all.merge(
    mang_cent.rename(columns={'geometry': 'mang_centroid'}),
    on='Mangrove_ID',
    how='left'
)
line_geoms = []
for row in conn.itertuples(index=False):
    if row.geometry is None or row.mang_centroid is None:
        line_geoms.append(None)
    else:
        line_geoms.append(LineString([row.geometry, row.mang_centroid]))

lines_num = geopandas.GeoDataFrame(
    conn[asset_key_cols + ['Avoided_EAD_USD', 'Mangrove_ID', 'nearest_mangrove_distance_m']].copy(),
    geometry=line_geoms,
    crs='EPSG:3448'
).dropna(subset=['geometry'])

# Numbered map (blue patches with numeric labels + connection lines)
fig, ax = plt.subplots(figsize=(16, 13))
ax.set_facecolor('white')
jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

# Context mangroves
mangroves.plot(ax=ax, facecolor='#d9d9d9', edgecolor='#8c8c8c', linewidth=0.20, alpha=0.50, zorder=2)

# Closest mangroves (blue)
mang_label.plot(ax=ax, facecolor='#007bff', edgecolor='#0b1f3a', linewidth=0.55, alpha=0.95, zorder=3)

# Keep lines so each asset can be linked to nearest mangrove
if len(lines_num) > 0:
    lines_num.plot(ax=ax, color='#ff7f0e', linewidth=0.45, alpha=0.35, zorder=4)

# Unattributed assets points (magenta)
asset_sizes = 16 + 10 * numpy.log10(numpy.clip(nearest_all['Avoided_EAD_USD'].values, 1e-6, None) + 1)
nearest_all.plot(ax=ax, color='#ff00aa', markersize=asset_sizes, alpha=0.88, zorder=5)

# Number labels next to mangrove patch centroids
label_points = mang_label.copy()
label_points['geometry'] = label_points.geometry.centroid
for row in label_points.itertuples(index=False):
    ax.text(
        row.geometry.x,
        row.geometry.y,
        str(int(row.Label_No)),
        fontsize=10,
        fontweight='bold',
        ha='center',
        va='center',
        color='black',
        bbox=dict(boxstyle='circle,pad=0.18', facecolor='white', edgecolor='black', linewidth=0.8, alpha=0.95),
        zorder=6,
    )

legend_handles = [
    Patch(facecolor='#d9d9d9', edgecolor='#8c8c8c', label='All FoN mangroves (context)'),
    Patch(facecolor='#007bff', edgecolor='#0b1f3a', label='Closest mangroves to unattributed assets'),
    Line2D([0], [0], color='#ff7f0e', lw=1.2, label='Asset -> nearest mangrove line'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff00aa', markeredgecolor='#ff00aa', markersize=8, label='Unattributed assets (Avoided_EAD_USD > 0)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='white', markeredgecolor='black', markersize=8, label='Number label (see reference table)')
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True, facecolor='white', framealpha=0.95)

ax.set_title(
    f'Numbered closest mangroves for unattributed positive avoided-EAD assets ({SCENARIO_FOR_ATTRIBUTION} scenario) | fixed {FIXED_BUFFER_M:,.0f} m buffer',
    fontsize=14,
)
ax.set_axis_off()
plt.tight_layout()

numbered_map_png = out_dir_attr / f'closest_mangroves_numbered_for_unattributed_assets_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png'
fig.savefig(numbered_map_png, dpi=300, bbox_inches='tight')
print(f'Saved: {numbered_map_png}')
plt.show()


In [ ]:
# Map gallery check: display all key map outputs
try:
    from IPython.display import Image, display
except Exception:
    Image = None

if 'out_dir_attr' not in globals():
    out_dir_attr = results_path_attr / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'

map_files = [
    out_dir_attr / f'assets_distance_threshold_bands_{MAP_SCENARIO}.png',
    out_dir_attr / f'mangrove_attribution_map_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
    out_dir_attr / f'mangrove_zero_attribution_and_unattributed_assets_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
    out_dir_attr / f'closest_mangroves_numbered_for_unattributed_assets_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
    out_dir_attr / f'all_unattributed_assets_and_closest_mangroves_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
    out_dir_attr / f'north_coast_unattributed_assets_and_closest_mangroves_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
    out_dir_attr / f'mangrove_attribution_hotspots_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
    out_dir_attr / f'mangrove_attribution_top10pct_hotspots_fixed_5000m_{SCENARIO_FOR_ATTRIBUTION}.png',
]

print('Map files present/missing:')
for p in map_files:
    status = 'FOUND' if p.exists() else 'MISSING'
    print(f'- [{status}] {p.name}')

if Image is not None:
    for p in map_files:
        if p.exists():
            print(f'\nDisplaying: {p.name}')
            display(Image(filename=str(p), embed=True))
